# Trace Count v4: v2-Style Steering

This notebook follows `pipeline_v4_steering_codex_prompt.md`. It intentionally reuses the v2 symbolic counting setup and v2-style HuggingFace GPT-2 architecture with learned absolute positional embeddings. The new part is mechanistic: hidden-state cache, probes, direction extraction, steering, and patching.

In [2]:
from pathlib import Path
import os
import pathlib
import subprocess
import sys

# Minimal Colab/VSCode setup. Keep dependency installs off unless an import check fails.
REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
REPO_DIR = Path("/content/Synthetic_CoT_NiaH_Count")
PULL_REPO = True
REPAIR_NUMPY_ABI = True
INSTALL_MINIMAL_DEPS = False
INSTALL_EDITABLE_PACKAGE = False

if Path("/content").exists():
    if REPO_DIR.exists():
        os.chdir(REPO_DIR)
        if PULL_REPO and (REPO_DIR / ".git").exists():
            subprocess.run(["git", "pull"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
        os.chdir(REPO_DIR)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from scripts.colab_setup import setup_colab

ROOT = setup_colab(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    pull=False,
    repair_numpy_abi=REPAIR_NUMPY_ABI,
    install_deps=INSTALL_MINIMAL_DEPS,
    install_editable=INSTALL_EDITABLE_PACKAGE,
)

numpy=2.0.2 pandas=2.2.2 scipy=1.16.3
cwd = /content/Synthetic_CoT_NiaH_Count
Dependency import check passed.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Runtime settings

- `debug`: quick end-to-end sanity check.
- `main`: v2-style main setting: two models, one seed by default, `1..10` counts, GPT-2 learned absolute positions.
- `stage='all'` runs training, behavior eval, hidden cache, probes, direction extraction, patching, steering, plots, and a markdown/html artifact report.

In [4]:
PRESET = 'main'  # 'debug' or 'main'
STAGE = 'all'
SEEDS = '1234'
OUT_ROOT = 'runs/synthetic_niah_v4'
RUN_NAME = ''  # empty means auto: {preset}_seed{seeds}
if not RUN_NAME:
    RUN_NAME = f"{PRESET}_seed{SEEDS.replace(',', '_')}"
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
SKIP_COMPLETED = True

print({
    'PRESET': PRESET,
    'STAGE': STAGE,
    'SEEDS': SEEDS,
    'OUT_ROOT': OUT_ROOT,
    'RUN_NAME': RUN_NAME,
    'DEVICE': DEVICE,
    'SKIP_COMPLETED': SKIP_COMPLETED,
})


{'PRESET': 'main', 'STAGE': 'all', 'SEEDS': '1234', 'OUT_ROOT': 'runs/synthetic_niah_v4', 'RUN_NAME': 'main_seed1234', 'DEVICE': 'cuda', 'SKIP_COMPLETED': True}


In [ ]:
cmd = [
    sys.executable, '-u', '-m', 'synthetic_niah_v4.run_v4',
    '--preset', PRESET,
    '--stage', STAGE,
    '--device', DEVICE,
    '--out-root', OUT_ROOT,
    '--seeds', SEEDS,
]
if RUN_NAME:
    cmd += ['--run-name', RUN_NAME]
if SKIP_COMPLETED:
    cmd.append('--skip-completed')
print(' '.join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
captured = []
for line in proc.stdout:
    print(line, end='')
    captured.append(line.rstrip())
returncode = proc.wait()
if returncode:
    print('---- Last 120 log lines ----')
    print('\n'.join(captured[-120:]))
    raise subprocess.CalledProcessError(returncode, cmd)

RUN_DIR = None
for line in reversed(captured):
    if line.startswith('FINAL_RUN_DIR '):
        RUN_DIR = Path(line.split(' ', 1)[1].strip())
        break
if RUN_DIR is None:
    RUN_DIR = Path(OUT_ROOT) / RUN_NAME if RUN_NAME else Path(OUT_ROOT)
print('RUN_DIR =', RUN_DIR)

/usr/bin/python3 -u -m synthetic_niah_v4.run_v4 --preset main --stage all --device cuda --out-root runs/synthetic_niah_v4 --seeds 1234 --run-name main_seed1234 --skip-completed
[v4] stage=train
[v4 train] non_thinking seed=1234

v4 non_thinking seed=1234: 100%|██████████| 10000/10000 [12:11<00:00, 13.67it/s, loss=0.0000, lr=8.94e-12]
[v4 train] thinking seed=1234

v4 thinking seed=1234: 100%|██████████| 10000/10000 [13:52<00:00, 12.01it/s, loss=0.0000, lr=8.94e-12]
[v4] stage=behavior_eval
[v4] stage=cache
[v4] stage=probe
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it wi

## Result tables

The most important tables are behavior accuracy, probe results, direction diagnostics, and steering/patching. Probe accuracy alone means decodability; steering or patching is the causal check.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

tables = RUN_DIR / 'tables'
for name in [
    'behavior_accuracy_by_count.csv',
    'probe_results.csv',
    'direction_metrics.csv',
    'steering_results.csv',
    'interchange_patching_results.csv',
]:
    path = tables / name
    display(Markdown(f'### `{name}`'))
    if path.exists() and path.stat().st_size > 0:
        display(pd.read_csv(path).head(20))
    else:
        display(Markdown('_No rows._'))

## Figures

These figures summarize: probe accuracy/R2 by anchor and layer, direction agreement with the count-token unembedding, steering dose response, steering controls, and patching from donor to receiver counts.

In [ ]:
from IPython.display import Image

for fig in sorted((RUN_DIR / 'figures').glob('*.png')):
    display(Markdown(f'### `{fig.name}`'))
    display(Image(filename=str(fig), width=900))

## Save to Google Drive

In [ ]:
SAVE_TO_DRIVE = False
DRIVE_SAVE_COMPLETED = False
DRIVE_DIR = '/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/Synthetic_CoT_NiaH_Count/colab_results/'

if SAVE_TO_DRIVE and Path('/content').exists():
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    dest = Path(DRIVE_DIR) / RUN_DIR.name
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(RUN_DIR, dest)
    DRIVE_SAVE_COMPLETED = True
    print('saved to', dest)
else:
    print('SAVE_TO_DRIVE is False; local RUN_DIR:', RUN_DIR)

## Auto-disconnect Colab Runtime

This cell runs immediately after the Google Drive save cell. It only disconnects when a Drive save was confirmed; local VSCode/Jupyter runs are not force-closed by default.

In [ ]:
AUTO_DISCONNECT_AFTER_DRIVE_SAVE = True
FORCE_LOCAL_KERNEL_SHUTDOWN = False

if AUTO_DISCONNECT_AFTER_DRIVE_SAVE and globals().get("DRIVE_SAVE_COMPLETED", False):
    import time

    print("Google Drive save completed. Flushing Drive and disconnecting Colab runtime in 3 seconds...")
    time.sleep(3)
    try:
        from google.colab import drive, runtime

        try:
            drive.flush_and_unmount()
            print("Google Drive flushed and unmounted.")
        except Exception as e:
            print(f"Drive flush/unmount skipped or failed: {e}")
        runtime.unassign()
    except Exception as e:
        print(f"Colab runtime disconnect unavailable or failed: {e}")
        if FORCE_LOCAL_KERNEL_SHUTDOWN:
            import IPython

            IPython.Application.instance().kernel.do_shutdown(restart=False)
        else:
            print("Not forcing local kernel shutdown.")
else:
    print("Auto-disconnect skipped: no confirmed Google Drive save, or AUTO_DISCONNECT_AFTER_DRIVE_SAVE is False.")

## Optional GitHub result upload

In [ ]:
PUSH_RESULTS_TO_GITHUB = False
GIT_BRANCH = 'v4-results'

if PUSH_RESULTS_TO_GITHUB:
    subprocess.run(['git', 'checkout', '-B', GIT_BRANCH], check=True)
    subprocess.run(['git', 'add', str(RUN_DIR)], check=True)
    subprocess.run(['git', 'commit', '-m', f'Add v4 results {RUN_DIR.name}'], check=False)
    subprocess.run(['git', 'push', '-u', 'origin', GIT_BRANCH], check=True)
else:
    print('PUSH_RESULTS_TO_GITHUB is False')